# Phase 2: Baselines + First Model

Wraps Phase 2 of the ROGII Wellbore Geology Prediction project. The work itself lives in `src/rogii_wellbore/` (testable code) and `scripts/run_*.py` (experiments logged to MLflow). This notebook is a *report*: it reads from MLflow and tells the story of what we built, what worked, what didn't, and what's next.

**Phase 2 objectives:**
1. Three constant-prediction baselines wired into `src/rogii_wellbore/models/constant.py`.
2. `evaluate.masked_rmse` confirmed to match competition semantics (eval-only rows, pooled across wells).
3. CV harness producing pooled OOF RMSE — reproduces prior-attempt carry-forward at 15.91.
4. First LightGBM model, residual-from-anchor target, OOF logged.
5. First submission to Kaggle, public LB noted.
6. All runs logged to `phase2` MLflow experiment.

**Key Phase 1 priors that shaped the design:**
- Eval zone is a contiguous tail (forward extrapolation, not infilling).
- MD step is uniform 1.0 ft; typewell on 0.5 ft step.
- Per-well GR normalization is appropriate (across-well-std / within-well-std ≈ 0.89).
- ~30% GR missingness; short NaN runs (linear interpolation safe).
- TVT is mostly flat through the eval tail — carry-forward beats linear-extrap by ~7x in the prior attempt. The right architecture is *anchor + small correction*, not free-form trajectory.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from rogii_wellbore.config import MLFLOW_TRACKING_URI

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print("python :", sys.executable)
print("mlflow :", MLFLOW_TRACKING_URI)

## 1. Constant baselines

Three heuristics, all per-well, no fitting:

- **Carry-forward** — predict the last known `TVT_input` value across the entire eval tail. The prior attempt reported pooled RMSE 15.91; this is the bar.
- **Smooth-anchor** — predict the *mean* of the last K known values (K ∈ {20, 50}). Hedges against a noisy last value.
- **Linear-extrap** — fit a line to (MD, TVT) on the last K=20 known rows and extrapolate. Tests whether trajectory information helps.

Each baseline runs through two CV schemes:
1. **Well-grouped** — `GroupKFold` by `well_id`. The standard.
2. **Pad-grouped** — KMeans-cluster the 773 wellhead X/Y points into 20 pads, `GroupKFold` by pad. Intended as a more pessimistic estimate (geographically close wells in the same fold).

In [ ]:
# Pull phase2 runs from MLflow and format the baselines table.
runs = mlflow.search_runs(experiment_names=["phase2"])

# Each run_name encodes baseline + CV scheme: e.g. "carry_forward" or "carry_forward__pad_cv".
runs = runs[
    runs["tags.mlflow.runName"].str.contains(
        "carry_forward|smooth_anchor|linear_extrap", regex=True
    )
].copy()
runs["cv"] = np.where(
    runs["tags.mlflow.runName"].str.endswith("__pad_cv"), "pad-grouped", "well-grouped"
)
runs["baseline"] = runs["tags.mlflow.runName"].str.replace("__pad_cv", "", regex=False)

# Latest run per (baseline, cv) — handles any earlier re-runs.
runs = runs.sort_values("start_time").groupby(["baseline", "cv"], as_index=False).last()

table = runs.pivot(index="baseline", columns="cv", values="metrics.oof_pooled_rmse").round(4)
table = table.reindex(
    ["carry_forward", "smooth_anchor_k20", "smooth_anchor_k50", "linear_extrap_k20"]
)
table

**Reading the table:**

- **Carry-forward is the bar at 15.91.** Reproduces the prior attempt exactly — confirms our CV + eval semantics are correct (this was the most important check of Phase 2).
- **Smoothing hurts.** k=20 loses 0.08; k=50 loses 0.22. The freshest known TVT is the best estimate; older values are stale. → Implies: a model should use the latest known TVT as anchor, not a window mean.
- **Linear extrapolation fails catastrophically (107.77 vs 15.91, ~7x worse).** Following the slope at the anchor amplifies error. TVT is approximately flat through the eval tail; the slope at the anchor is noise. → Implies: predict *anchor + small correction*, not a trajectory.
- **Pooled RMSE is identical across CV schemes.** Expected: carry-forward is a per-well computation; reshuffling fold composition doesn't change which value goes where. Per-fold spread *does* change — see next section.

## 2. Per-fold variance and CV scheme comparison

Pooled RMSE doesn't change between CV schemes, but per-fold RMSE does, and the per-fold *spread* tells us how much we'd swing under unlucky fold composition. This matters for interpreting any future improvement: if a new model beats carry-forward by 0.3 RMSE pooled, but fold variance is 4 RMSE, the improvement is below the noise floor.

In [ ]:
# Per-fold RMSE for carry-forward under both CV schemes.
fold_cols = [f"metrics.oof_fold{i}_rmse" for i in range(5)]
cf = runs[runs["baseline"] == "carry_forward"].set_index("cv")[fold_cols]
cf.columns = [f"fold {i}" for i in range(5)]
cf

In [ ]:
# Plot per-fold RMSE, side by side.
fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(5)
w = 0.38
ax.bar(x - w / 2, cf.loc["well-grouped"].values, width=w, label="well-grouped", color="#4472c4")
ax.bar(x + w / 2, cf.loc["pad-grouped"].values, width=w, label="pad-grouped", color="#ed7d31")
ax.axhline(15.91, ls="--", color="black", lw=1, label="pooled RMSE (15.91)")
ax.set_xlabel("fold")
ax.set_ylabel("carry-forward RMSE")
ax.set_title("Per-fold carry-forward RMSE under two CV schemes")
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.show()

spread_well = cf.loc["well-grouped"].max() - cf.loc["well-grouped"].min()
spread_pad = cf.loc["pad-grouped"].max() - cf.loc["pad-grouped"].min()
print(f"Spread (max − min): well-grouped = {spread_well:.2f}, pad-grouped = {spread_pad:.2f}")  # noqa: RUF001

**Surprise:** pad-grouped CV produces a *smaller* per-fold spread, not larger. We expected pad-grouping to give a more pessimistic estimate by holding out geographically clustered wells. It didn't.

The interpretation: the field is geographically homogeneous enough that pad assignment effectively reshuffles wells across folds without creating systematically harder holdouts. Each pad-grouped fold is a more representative sample of the population than each well-grouped fold (where one fold might land on a bunch of long-eval wells and another on short-eval wells).

**Implications going forward:**
- The 4.2 RMSE well-grouped spread is the realistic "fold variance" we need to beat to claim improvement.
- A model improvement < 1.5 RMSE pooled is not reliably distinguishable from fold variance without multi-seed runs.
- We report both CV schemes, but **well-grouped pooled is the headline number** since it's more conservative.

## 3. First LightGBM model

The constant baselines establish two facts: (a) carry-forward at 15.91 is a strong bar, and (b) any trajectory-following model gets crushed (linear-extrap at 107). So the right architecture is **anchor + small correction**.

**Design:**

- **Target.** Residual from anchor: `y = TVT - anchor_TVT`. The model can predict zero correction and recover carry-forward exactly. Predictions are added back to anchor at inference.
- **Anchor (inference).** The last known `TVT_input` row of the well — the natural cutoff between known and eval segments.
- **Anchor (training).** A synthetic anchor inside the known segment, parameterized by `frac ∈ {0.95, 0.90, 0.85, 0.80}`. **Multi-anchor** means each well contributes 4 training examples with different anchor positions, exposing the model to anchor variability without straying far from where the inference anchor will be.
- **Features (v1, 4 columns):** `dmd` (MD - anchor_MD), `dz` (Z - anchor_Z), `gr_roll_mean_k=50` (causal rolling mean of GR over last 50 rows), `gr_roll_std_k=50` (causal rolling std).
- **Validation.** GroupKFold by well_id, 5 splits. 10% of each fold's training wells held out for LightGBM internal early stopping.

Code lives in `src/rogii_wellbore/models/lgbm.py` and `src/rogii_wellbore/features.py`. Run via `scripts/run_lgbm.py`.

### 3.1 The residual-distribution bug (RMSE 80 → 15.42)

The first version of this model — single training anchor at `frac=0.4` with `anchor_tvt`/`anchor_z` as features — produced a pooled OOF RMSE of **80.80**, *five times worse* than carry-forward. Best iterations across folds were 48–146 (out of 2000), meaning the model was overfitting very fast — symptom of weak signal and a training/inference distribution shift.

A targeted diagnostic (see `scripts/diagnose_lgbm.py`) revealed two problems:

In [ ]:
# Diagnostic numbers captured from scripts/diagnose_lgbm.py (50 val wells, fold 0).
# Reproduced here for the narrative — these are the smoking gun.
diag = pd.DataFrame(
    {
        "dmd_min": [1, 1],
        "dmd_median": [2986, 2472],
        "dmd_max": [9539, 8550],
        "residual_mean": [+123.56, +3.06],
        "residual_std": [66.69, 14.56],
        "residual_min": [-6.4, -40.0],
        "residual_max": [+305.4, +53.4],
    },
    index=["training (anchor=0.4)", "inference (anchor=1.0)"],
)
diag

**Bug 1 — residual distribution mismatch.** Training residuals had mean **+123.56** and ranged from **-6 to +305**. Inference residuals had mean **+3.06** and ranged from **-40 to +53**. The model was trained on a completely different target distribution from the one it had to predict. It learned "predict +30 to +100 on average" and applied that at inference, where the truth was ~0.

Root cause: with the training anchor at 40% of the known segment, post-anchor rows included the entire eval segment, and TVT (in absolute thousand-foot values) drifts by 100+ over 5000 ft of MD. At inference, the anchor is at 100% of the known segment, so the model only needs to predict a small drift over the remaining eval rows.

**Bug 2 — leaky "well-identity" features.** `anchor_tvt` and `anchor_z` were the #1 and #3 features by gain, but they're *constant within a well* and they take *different values* between training (anchor=0.4) and inference (anchor=1.0) for the same well. The model partitioned training data using "well identity" via these features, then applied the wrong partition's rules at inference.

**Fix (two changes):**
1. **Narrow `anchor_fracs` to `(0.95, 0.90, 0.85, 0.80)`** — every training anchor now corresponds to a position close to inference, so training-residual distribution matches inference-residual distribution.
2. **Drop `anchor_tvt` and `anchor_z` from the feature set.** Keep only features that vary row-to-row within a well: `dmd`, `dz`, `gr_roll_mean_k`, `gr_roll_std_k`.

Pooled OOF RMSE after the fix: **15.42** — beats carry-forward by 0.49.

### 3.2 Result

LGBM v1 (the fixed version) was logged to MLflow as `lgbm_v1` in the `phase2` experiment.

In [ ]:
# LGBM v1 pulled from MLflow.
all_runs = mlflow.search_runs(experiment_names=["phase2"])
lgbm = all_runs[all_runs["tags.mlflow.runName"] == "lgbm_v1"].sort_values("start_time").iloc[-1]

cf_run = runs[(runs["baseline"] == "carry_forward") & (runs["cv"] == "well-grouped")].iloc[0]

fold_cols = [f"metrics.oof_fold{i}_rmse" for i in range(5)]
cf_folds = cf_run[fold_cols].to_numpy(dtype=float)
lgbm_folds = lgbm[fold_cols].to_numpy(dtype=float)

comp = pd.DataFrame(
    {
        "carry_forward": cf_folds,
        "lgbm_v1": lgbm_folds,
        "delta (cf − lgbm)": cf_folds - lgbm_folds,  # noqa: RUF001
    },
    index=[f"fold {i}" for i in range(5)],
)
comp.loc["pooled"] = [
    cf_run["metrics.oof_pooled_rmse"],
    lgbm["metrics.oof_pooled_rmse"],
    cf_run["metrics.oof_pooled_rmse"] - lgbm["metrics.oof_pooled_rmse"],
]
comp.round(4)

In [ ]:
# Per-fold delta plot.
fig, ax = plt.subplots(figsize=(7, 3.5))
delta = cf_folds - lgbm_folds
colors = ["#70ad47" if d > 0 else "#c00000" for d in delta]
ax.bar(range(5), delta, color=colors)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(5))
ax.set_xticklabels([f"fold {i}" for i in range(5)])
ax.set_ylabel("RMSE improvement (carry-forward − LGBM v1)")  # noqa: RUF001
ax.set_title("LGBM v1 beats carry-forward in 5/5 folds")
plt.tight_layout()
plt.show()

**Interpretation.** Pooled improvement of 0.49 RMSE is *below* the fold-variance noise floor (~4 RMSE), but the **5/5 per-fold win** is the stronger signal: every fold, regardless of which wells it holds out, the LGBM beats carry-forward by a small but consistent amount. That consistency rules out luck-of-the-draw on fold composition.

The architecture works. The features are starved: 4 columns (`dmd`, `dz`, two rolling-GR stats) is the minimum that can possibly beat anchor-only. The headline lever for the next phase is obvious — the typewell. We have a per-well vertical GR-vs-TVT reference profile sitting unused.

**LightGBM best iterations across folds:** 43, 14, 25, 41, 27. Healthy — no fold ran away with 487 rounds chasing noise (which was the v1-broken symptom). Early stopping kicks in cleanly within ~50 rounds, indicating the 4-feature signal saturates quickly.

## 4. Kaggle submission

Trained one LightGBM per fold (5 models) on the full train set, pickled them, and uploaded to a Kaggle dataset. The inference notebook (Kaggle-side) featurizes the 3 test wells using the same `build_features_for_well` logic as training, averages predictions across the 5 fold models, adds the per-well anchor back to convert residual → absolute TVT, and writes `submission.csv` (14,151 rows).

| Submission | Pooled OOF | Public LB | Notes |
|---|---|---|---|
| Carry-forward (Phase 1 baseline) | 15.9099 | **14.505** | submitted at end of Phase 1 |
| LGBM v1 (this phase) | 15.4199 | **15.142** | 4-feature residual model |
| **Δ (LGBM − CF)** | **−0.49 (better)** | **+0.64 (worse)** | — |

**OOF-LB inversion.** The model that beats carry-forward on OOF *loses* to carry-forward on the LB. This is not a sign that the OOF is broken — both numbers are correct measurements of what they measure. They're measuring different populations:

- **OOF** averages over 773 wells distributed across the entire field.
- **LB** is computed on 3 specific test wells, all in the densest part of the NE cluster (per Phase 1 EDA).

The 3 test wells happen to be *unusually CF-friendly* — TVT is even flatter through their eval tails than the cross-well average. When the truth is "no correction," any non-zero LGBM residual prediction adds error. The model's strength (small consistent corrections) becomes a liability on a sample where the correct answer is "predict zero correction."

Note also: on LB, CF lands at 14.505 versus its OOF 15.91 — meaning the 3 test wells are easier than the average of 773 wells by ~1.4 RMSE *for carry-forward alone*. Our LGBM only sees a 0.27 LB improvement over its OOF (15.42 → 15.14), so it benefits less from the easier sample than CF does. The 4-feature model isn't differentiated enough to exploit easy wells.

## 5. Phase 3 plan

**The gap is the typewell.** The dataset includes a per-well vertical reference profile (`TVT, GR, Geology` at 0.5 ft step), which is literally "what TVT looks like at each MD if you projected down vertically." Our current model uses none of this. The prior attempt's `predict_gr_match_v2` (sliding-window cosine similarity between lateral GR and typewell GR-vs-TVT template) produced a per-row TVT estimate from the typewell — exactly the missing signal.

**First Phase 3 move:** port `predict_gr_match_v2` from `archive/prior_attempt/features.py` into `src/rogii_wellbore/features.py`. Add two features to LGBM v2: `gr_match_tvt` (typewell-derived TVT prediction) and `gr_match_sim` (similarity score; signals when the match is unreliable). A/B test against v1 on OOF. If it gives ≥1 RMSE improvement, submit and check LB.

**Secondary levers ordered by expected impact:**
1. **Typewell features** (above) — single highest-information signal in the dataset.
2. **Per-well GR z-score** — Phase 1 found across-well GR variation is real (per-pad bimodality). A/B test as a feature switch.
3. **Pad-cluster ID** — categorical feature, captures "which pad" patterns that may correlate with TVT zones.
4. **Multi-seed runs** — log fold variance, average across seeds. Needed before claiming small improvements.
5. **Sequence model** (LSTM/transformer on a windowed view) — Phase 4. Requires the typewell baseline first.

**What to do about the 4 high-NaN-GR wells and 11 below-typewell-coverage wells:** evaluate v2 with and without; decide whether to drop, downweight, or special-case.

**Phase 2 ends here.** Tag `v0.2-baselines`, push, move to Phase 3.